# Lesson 1 — REINFORCE, advantages, and baselines

You already know supervised learning. Here we learn from **sampled actions and scalar rewards**, without being given the correct action on each example.

This notebook uses complete implementations in `src/rl_study/`. First run it unchanged, then predict what an ablation will do before changing the controls.

By the end, you should be able to trace a reward to a policy gradient and explain what a baseline changes. No GPU or third-party packages are needed for the calculations. Jupyter is needed for the notebook UI.

In [ ]:
from pathlib import Path
import sys

# Works when launched from the repository root or its notebooks directory.
candidates = [Path.cwd(), *Path.cwd().parents]
ROOT = next((p for p in candidates if (p / "src" / "rl_study" / "bandit.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Launch this notebook from inside the learning_RL repository.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from rl_study.bandit import (
    BASELINES, Config, REWARD_PROBS, PythonAgent, softmax,
    exact_gradient, exact_estimator_stats, frozen_variance_experiment, train,
)
from rl_study.report import learning_curves_svg
import statistics

print("Repository:", ROOT)

## The environment and policy

A context $c$ is drawn uniformly from three possibilities. The policy samples an action $a$ from three possibilities. The environment returns a Bernoulli reward with probability $q_{c,a}$.

| Context | Action 0 | Action 1 | Action 2 |
|---|---:|---:|---:|
| 0 | 0.85 | 0.50 | 0.30 |
| 1 | 0.05 | 0.75 | 0.10 |
| 2 | 0.25 | 0.10 | 0.90 |

Each context has a different optimal action. Their average optimal expected reward is $(0.85+0.75+0.90)/3=0.8333$.

Our policy is $\pi_\theta(a\mid c)=\mathrm{softmax}(\theta_c)_a$. It starts uniform. The logits are a trainable table, equivalent to a linear layer on a one-hot context. The critic is one trainable value per context.

This is a **contextual bandit**, not a sequential environment. There is no discounting, bootstrapping, or temporal credit assignment yet. In an LLM, a completion consists of many token actions; we will add that structure later.

In [ ]:
initial_agent = PythonAgent()
initial_probabilities = initial_agent.probabilities()
optimal_reward = sum(max(row) for row in REWARD_PROBS) / 3
print("Initial policy:", initial_probabilities)
print(f"Optimal expected reward: {optimal_reward:.4f}")

## From expected reward to REINFORCE

We maximize
$$J(\theta)=\mathbb E_{c,a,r}[r].$$

Using $\nabla\pi=\pi\nabla\log\pi$ gives
$$\nabla J=\mathbb E[(r-b(c))\nabla\log\pi_\theta(a\mid c)].$$

For a batch of $B$ samples, gradient descent minimizes the surrogate
$$L_\text{policy}=-\frac1B\sum_i\operatorname{stopgrad}(r_i-b(c_i))\log\pi_\theta(a_i\mid c_i).$$

The samples and rewards are constants in this backward pass. We do not differentiate through sampling. The surrogate loss is a way to obtain a gradient estimate; its numerical value is not the expected reward.

For our softmax table:
$$\frac{\partial\log\pi(a\mid c)}{\partial\theta_{c,j}}=\mathbf1[j=a]-\pi(j\mid c).$$

The Python backend writes this derivative directly and performs **gradient ascent** on reward. The PyTorch backend minimizes the negative objective. These have the same sign.

Before running the next cell: what should happen to the probability of action 1 when its advantage is positive? Negative?

In [ ]:
for reward, baseline in [(1.0, 0.0), (0.0, 1.0)]:
    agent = PythonAgent()
    before = agent.probabilities()[0][1]
    agent.update([(0, 1, reward)], [baseline] * 3,
                 policy_lr=0.1, value_lr=0.1, learn_value=False)
    after = agent.probabilities()[0][1]
    print(f"Advantage {reward - baseline:+.1f}: P(action 1) {before:.4f} -> {after:.4f}")

## Why subtracting a baseline works

For a baseline that depends only on context, its expected score gradient is zero:
$$\mathbb E_{a\sim\pi}[b(c)\nabla\log\pi(a\mid c)]
=b(c)\nabla\sum_a\pi(a\mid c)=0.$$

So a baseline can change estimator variance without changing its expected gradient. We detach it because differentiating through $b$ introduces an extra term outside this identity.

Our three variants:

- **None:** $b=0$.
- **Running:** an exponential moving average of rewards from **previous** batches.
- **Learned:** $b(c)=V_\phi(c)$, trained on $\frac12\operatorname{mean}(V_\phi(c)-r)^2$.

The policy uses the previous baseline, then the baseline is updated. Subtracting the current batch mean including each sample's own reward would bias this plain estimator. Group-relative methods have their own estimator details, which we will study later.

Because the environment is tiny, we can enumerate all contexts, actions, and rewards. Check the expected gradient for arbitrary context baselines against the exact derivative of $J$.

In [ ]:
probabilities = [softmax(row) for row in ((0.4, -0.2, 0.7), (-1, 2, 0), (1, 0.4, 0.3))]
target = exact_gradient(probabilities)
for baseline in ([0, 0, 0], [0.4, 0.4, 0.4], [0.2, 0.8, -0.1]):
    mean, variance = exact_estimator_stats(probabilities, baseline, batch_size=64)
    error = max(abs(mean[c][a] - target[c][a]) for c in range(3) for a in range(3))
    assert error < 1e-12
    print(f"baseline={baseline}: gradient error={error:.2e}, variance={variance:.6f}")

## Measure variance without moving the policy

Comparing variance at different points of different learning curves confounds baseline effects with policy changes. Here we hold the policy uniform and reuse identical sampled batches for each estimator.

For this diagnostic, the running baseline is the exact global mean reward and the learned baseline is the exact context value: their **converged** values, rather than their initial zeros. This isolates the effect of baseline quality.

We measure $\operatorname{tr}\operatorname{Cov}(g)$, the sum of variances of all gradient coordinates. Averaging $B$ independent samples divides this variance by $B$. The empirical mean gradient should approach the analytic gradient.

A value baseline is useful, but it is not generally the exact variance-minimizing baseline: that also depends on the squared norm of the score gradient.

In [ ]:
variance_results = frozen_variance_experiment(batch_size=64, repetitions=2000)
print(f"{'baseline':<10} {'exact variance':>16} {'empirical':>14} {'gradient error':>16}")
for result in variance_results:
    print(f"{result['method']:<10} {result['exact_variance']:>16.6f} "
          f"{result['empirical_variance']:>14.6f} {result['max_gradient_error']:>16.6f}")

## Train all three variants

Use the same sample budget and five seeds. An identical seed gives reproducible random draws, although the sampled actions can diverge as policies diverge.

The metrics use the **known reward probabilities** to evaluate expected reward exactly, rather than another noisy batch. Entropy measures policy randomness. The variance curve enumerates the estimator variance at the current policy and baseline; use the previous frozen-policy experiment to isolate baseline effects.

The shaded bands show one standard deviation across seeds, not confidence intervals. This easy task can give very similar final policies for all variants. Lower estimator variance does not guarantee a visibly better final reward.

In [ ]:
config = Config(steps=1000, batch_size=64, policy_lr=0.5, value_lr=0.5,
                seeds=(0, 1, 2, 3, 4), backend="python")
rows = []
for method in BASELINES:
    for seed in config.seeds:
        rows.extend(train(config, method, seed))

for method in BASELINES:
    final = [row for row in rows if row["method"] == method and row["step"] == config.steps]
    mean_reward = statistics.mean(row["expected_reward"] for row in final)
    sd_reward = statistics.stdev(row["expected_reward"] for row in final)
    mean_optimal = statistics.mean(row["optimal_action_probability"] for row in final)
    print(f"{method:<8}: reward={mean_reward:.4f} ± {sd_reward:.4f}, "
          f"P(optimal)={mean_optimal:.4f}")
assert all(row["expected_reward"] > 0.8 for row in rows if row["step"] == config.steps)

In [ ]:
svg = learning_curves_svg(rows)
try:
    from IPython.display import SVG, display
except ImportError:
    print("Plot computed. Run the CLI to save learning_curves.svg, or open this in Jupyter.")
else:
    display(SVG(svg))

## Compare with autograd

The optional PyTorch implementation uses the same rollout batch, advantage, and SGD step. Read the short implementation below next to `score_gradient` and `PythonAgent.update`.

The critic is optimized separately. Policy loss must not backpropagate through the baseline or reward. Our cross-backend test checks numerical update equivalence when PyTorch is installed.

If PyTorch is unavailable, this cell skips the demo; all previous calculations remain complete.

In [ ]:
import importlib.util
if importlib.util.find_spec("torch") is None:
    print("Optional PyTorch backend is not installed. See README.md for setup.")
else:
    import inspect
    from rl_study.torch_bandit import TorchAgent
    print(inspect.getsource(TorchAgent.update))
    batch = [(0, 0, 1.0), (1, 2, 0.0), (2, 2, 1.0)]
    explicit, autograd = PythonAgent(), TorchAgent()
    for agent in (explicit, autograd):
        agent.update(batch, [0.2, 0.4, 0.6], 0.5, 0.5, True)
    error = max(abs(x - y) for a, b in zip(explicit.probabilities(), autograd.probabilities())
                for x, y in zip(a, b))
    assert error < 1e-12
    print("Explicit vs. autograd max probability difference:", error)

## Experiments to do next

1. Set `policy_lr` to 0.1, then 1.0. Predict which effect you expect before running.
2. Set `batch_size` to 8, then 128. Adjust `steps` to keep `steps * batch_size` fixed.
3. Set `value_lr` very low. What happens when the critic lags behind the policy?
4. Inspect the frozen-policy variance and the learning curves together. Explain why they answer different questions.

Write down answers to these questions:

- Why does a positive reward sometimes give a negative advantage?
- Why do the other action logits change after only one action was sampled?
- What breaks if the baseline depends on the sampled action?
- Why is the critic predicting expected reward rather than identifying the optimal action?

**Connection to LLMs:** the prompt becomes context; generated tokens are actions; the policy is a causal LM. A response-level reward can weight the sum of its token log probabilities. GRPO introduces a baseline relative to several responses to a prompt; PPO usually learns a critic; offline DPO uses fixed preference pairs. We will implement those differences after establishing a shared LLM task and evaluator.

Next lesson: [LLM evaluation and SFT](../STUDY_PLAN.md).